In [267]:
using LowLevelFEM, LinearAlgebra, SparseArrays

In [268]:
openGeometry("beam-2.geo")

mat = Material("beam")
U = Field([mat], type=:VectorField, dim=2, fieldName=:u, rhsName=:f)
Φ = Field([mat], type=:ScalarField, dim=2, fieldName=:φ, rhsName=:m, reducedOrder=true);

Info    : Clearing all models and views...
Info    : Done clearing all models and views
Info    : Reading 'beam-2.geo'...
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 60%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Circle)
Info    : Done meshing 1D (Wall 0.000565835s, CPU 0.000566s)
Info    : Meshing order 2 (curvilinear on)...
Info    : [  0%] Meshing curve 1 order 2
Info    : [ 30%] Meshing curve 2 order 2
Info    : [ 60%] Meshing curve 3 order 2
Info    : [ 80%] Meshing curve 4 order 2
Info    : Done meshing order 2 (Wall 0.00071473s, CPU 0.000714s)
Info    : 114 nodes 62 elements
Info    : Done reading 'beam-2.geo'


In [269]:
b = 30
h = 5
E = mat.E
G = mat.μ
Iz = b * h^4 / 12
κ = 2 / 3
A = b * h;

In [270]:
t = tangentVector(U, "beam")
tx, ty, tz = t[1], t[2], t[3]

nx = -ty
ny = tx
nz = 0;

In [271]:
# ------------------------------------------------------------------
# Generalized Timoshenko beam strain
#
# ε = tᵀ ∇u t
# γ = nᵀ ∇u t - φ
# κ = ∇φ ⋅ t
# ------------------------------------------------------------------

Au = [
    tx*tx tx*ty tx*ty ty*ty
    nx*tx nx*ty ny*tx ny*ty
    0 0 0 0
]

Aφ = [
    0 0
    0 0
    tx ty
]

Gφ = [
    0; -1; 0;;
]

Bu = Au ⋅ Grad(U)

Bφ = Aφ ⋅ Grad(Φ) + Gφ ⋅ Φ;

In [272]:
D = [
    E*A 0 0
    0 κ*G*A 0
    0 0 E*Iz
]

3×3 Matrix{Float64}:
 3.0e7  0.0        0.0
 0.0    7.69231e6  0.0
 0.0    0.0        3.125e8

In [273]:
B = Bu + Bφ
K = ∫(B' ⋅ D ⋅ B, Γ="beam");

In [274]:
supp_u = BoundaryCondition("A", field=U, ux=0, uy=0)
supp_φ = BoundaryCondition("A", field=Φ, φ=0);

In [275]:
fu = ∫(U ⋅ [1, 0], Γ="C")
fφ = ∫(Φ ⋅ 0, Γ="B1")
fu2 = ∫(U ⋅ [0.0, -0.0], Γ="beam");

In [276]:
F = SystemVector([fu + fu2, fφ]);

In [277]:
bc1 = BoundaryCondition("O", field=U, ux=0, uy=0)
bc2 = BoundaryCondition("O", field=Φ, φ=0)

BoundaryCondition("O", Problem("beam-2", :ScalarField, 2, 1, Material[Material("beam", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 114, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :m, true), Dict{Symbol, Union{Function, Number, ScalarField}}(:φ => 0))

In [278]:
u, φ = solveField(K, F, support=[supp_u, supp_φ, bc1, bc2]);

In [279]:
showDoFResults(fu + fu2, name="force")
u0 = showDoFResults(u, name="u", factor=1, visible=true)
φ0 = showDoFResults(φ, name="φ");

In [280]:
n = VectorField([nx, ny, 0nx]);

In [281]:
g = grad(expandTo3D(u))

ε = t ⋅ (g * t)
γ = n ⋅ (g * t) - nodesToElements(φ)

κb = grad(φ) ⋅ t

N = E * A * ε
T = κ * G * A * γ
Mh = E * Iz * κb;

In [282]:
N0 = showElementResults(N, name="N")
T0 = showElementResults(T, name="T")
Mh0 = showElementResults(Mh, name="Mh")

5

In [283]:
plotOnBeam("beam", N, name="N graph")
plotOnBeam("beam", elementsToNodes(N), name="N graph")
plotOnBeam("beam", T, name="T graph")
plotOnBeam("beam", elementsToNodes(T), name="T graph")
plotOnBeam("beam", Mh, name="Mh graph")
plotOnBeam("beam", elementsToNodes(Mh), name="Mh graph")

11

In [284]:
openPostProcessor()

-------------------------------------------------------
Version       : 4.15.2-git
License       : GNU General Public License
Build OS      : Linux64-sdk
Build date    : 19700101
Build host    : amdci7.julia.csail.mit.edu
Build options : 64Bit ALGLIB[contrib] ANN[contrib] Bamg Blossom Cairo DIntegration Dlopen DomHex Eigen[contrib] Fltk GMP Gmm[contrib] Hxt Jpeg Kbipack LinuxJoystick MathEx[contrib] Mesh Metis[contrib] Mmg Mpeg Netgen Nii2mesh ONELAB ONELABMetamodel OpenCASCADE OpenCASCADE-CAF OpenGL OpenMP OptHom Parser Plugins Png Post QuadMeshingTools QuadTri Solver TetGen/BR TinyXML2[contrib] Untangle Voro++[contrib] WinslowUntangler Zlib tinyobjloader
FLTK version  : 1.3.8
OCC version   : 7.9.2
Packaged by   : root
Web site      : https://gmsh.info
Issue tracker : https://gitlab.onelab.info/gmsh/gmsh/issues
-------------------------------------------------------


In [285]:
@show maximum(u[1].a)
@show minimum(u[1].a)

maximum((u[1]).a) = 0.006809855792883714
minimum((u[1]).a) = -0.0005163456662709072


-0.0005163456662709072